# Bronze — Budgets

Lê o CSV de `landing/budgets/` correspondente ao `execution_date` e faz `append` no Delta Lake.

**Estratégia:** `append` — a deduplicação por `id` ocorre na Silver.  
**Nota:** o campo `year/month` do budget é o período orçamentário, independente do `execution_date`.

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15

In [ ]:
import sys
import os

year, month, day = int(year), int(month), int(day)

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if os.path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DecimalType, TimestampType
from pyspark.sql.utils import AnalysisException

In [ ]:
spark = create_spark_session("bronze_budgets")

# Arquivo gerado pela landing para este execution_date
input_path = f"s3a://datalake/landing/budgets/budgets_{execution_date}.csv"

try:
    df = spark.read.csv(input_path, header=True, inferSchema=False)
except AnalysisException:
    print(f"[bronze_budgets] Arquivo não encontrado: {input_path}. Sem dados novos.")
    spark.stop()
    df = None

if df is not None:
    df_bronze = (
        df.select(
            F.col("id").cast(IntegerType()).alias("id"),
            F.col("team_id").cast(IntegerType()).alias("team_id"),
            F.col("year").cast(IntegerType()).alias("year"),
            F.col("month").cast(IntegerType()).alias("month"),
            F.col("provider").cast(StringType()).alias("provider"),
            F.col("amount_usd").cast(DecimalType(12, 2)).alias("amount_usd"),
            F.col("created_at").cast(TimestampType()).alias("created_at"),
        )
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_layer", F.lit("landing"))
    )

    output_path = "s3a://datalake/bronze/budgets/"
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .save(output_path)
    )

    print(f"[bronze_budgets] {df_bronze.count()} linhas → {output_path}")
    df_bronze.groupBy("year", "month").count().orderBy("year", "month").show()
    spark.stop()